# 第50课 · 用 13 个数字听懂一帧声音——MFCC 完整流水线（STFT → Mel → log → DCT）

**目标**：串联所有 DSP 步骤，手工实现完整 MFCC 提取，结果与 `aurora.audio.mfcc.mfcc()` 数值对齐。

🔗 Aurora 连接：`aurora.audio.mfcc.mfcc()`、`aurora.audio.mfcc.dct_ii()`、`aurora.audio.mel.mel_spectrogram()`；这套特征是 Month 2 关键词识别分类器的标准输入。

← **上一课**　[L49 · DCT-II 离散余弦变换](L49_dct.ipynb)

> 上节课学习了 **DCT-II 离散余弦变换**：去相关（decorrelation）原理，纯 NumPy 实现替代 scipy.fft.dct。  
> 本课将探讨 **MFCC 完整流水线**。

## 本课剧情：手机语音助手如何用 13 个数字认识你的声音？

你说一个字，手机麦克风录下约 0.03 秒（30ms）的声音——大约 480 个采样点。这 480 个数字是原始 PCM，含有大量冗余和噪声。

语音识别模型的实际输入不是这 480 个数字，而是 **MFCC 的 13 个系数**——用 5 步流水线把冗余压缩殆尽：

```
x      (T,)                    ← 原始音频
  ↓ STFT + 功率谱
P      (n_frames, n_fft//2+1)  ← 每帧功率谱（去掉相位）
  ↓ × mel_filterbank
M      (n_frames, n_mels)      ← Mel 能量（模拟人耳）
  ↓ log(·+ε)
logM   (n_frames, n_mels)      ← 对数压缩（模拟响度感知）
  ↓ DCT-II
MFCC   (n_frames, n_mfcc)      ← 去相关倒谱系数 ✓
```

**每步的物理意义**：

| 步骤 | 操作 | 去掉了什么 | 保留了什么 |
|---|---|---|---|
| STFT | 分帧+FFT | 时域波形细节 | 频率成分 |
| Mel | 三角滤波器 | 高频冗余 | 人耳感知频率 |
| log | 取对数 | 绝对响度 | 相对响度差异 |
| DCT | 余弦变换 | 通道间相关性 | 独立的谱包络参数 |

本节任务：实现 `my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)`，与 `aurora.audio.mfcc.mfcc` 误差 < 1e-5。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine

## 1. 公式全景

```
MFCC[n] = DCT-II( log( mel_filterbank @ |STFT|² ) )[:n_mfcc]
```

- `|STFT|²` 是每帧的**功率谱（power spectrum）**，形状 `(n_frames, n_fft//2+1)`
- `mel_filterbank @ |STFT|²` 将功率谱映射到 mel 频带，形状 `(n_frames, n_mels)`
- `log(...)` 压缩动态范围（幅度相差 100 倍 → dB 只差 20）
- `DCT-II` 去相关，系数从低到高对应从粗到细的谱包络信息
- 取前 `n_mfcc` 个系数（`[:n_mfcc]`）；k=0 是所有 mel 能量的直流均值，保留它与 librosa 默认行为一致

对于正交归一化（`norm="ortho"`）的 DCT-II，第 `k` 个系数为：

```
y[k] = w[k] * sum_{n=0}^{N-1} x[n] * cos( pi/N * (n + 0.5) * k )
```

其中 `w[0] = sqrt(1/N)`，`w[k>0] = sqrt(2/N)`。

In [ ]:
# 演示：DCT-II 去相关的效果
from aurora.audio.mfcc import dct_ii

sr = 16000
x = sine(440, duration=0.5, sample_rate=sr)

# 构造一帧 mel 对数谱（40 mel bins）
from aurora.audio.mel import mel_spectrogram, power_to_db
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
# 取第 5 帧示例
frame = log_mel[5]
dct_frame = dct_ii(frame)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(frame)), frame)
axes[0].set_title("log-mel 谱（40 bins，第5帧）")
axes[0].set_xlabel("mel bin")
axes[0].set_ylabel("dB")
axes[1].bar(range(len(dct_frame)), dct_frame, color="orange")
axes[1].axvline(12.5, color="red", linestyle="--", label="截止：保留 k=0..12，共 n_mfcc=13 个")
axes[1].set_title("DCT-II 系数（取 k=0..12，n_mfcc=13 含 k=0）")
axes[1].set_xlabel("DCT 系数索引 k")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"k=0（直流）: {dct_frame[0]:.3f}  k=1..4: {dct_frame[1:5].round(3)}")

## 2. 为什么 n_mfcc 通常取 13？

语音的谱包络可以用很少的参数描述：低阶 DCT 系数对应声道形状（元音特征），高阶系数对应细节（声门激励、噪声）。实验表明：

| n_mfcc | 适用场景 |
|--------|---------|
| 13 | 经典 MFCC，GMM-HMM 语音识别标准配置 |
| 20–40 | 深度学习分类器，信息量更足 |
| > 40 | 收益递减；等同于直接用 log-mel |

`n_mels` 控制 mel 滤波器组的分辨率（40–128 常见）；`n_mfcc` ≤ `n_mels`。增大 `n_mfcc` 相当于保留更多高阶谱细节，同时增加过拟合风险。

In [ ]:
# 演示：不同 n_mfcc 下的系数能量分布
from aurora.audio.mfcc import dct_ii
from aurora.audio.mel import mel_spectrogram, power_to_db

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
all_coeffs = dct_ii(log_mel)  # (n_frames, 40)

energy_per_coeff = np.mean(all_coeffs**2, axis=0)  # 每个系数在所有帧上的均方值

plt.figure(figsize=(8, 3))
plt.bar(range(len(energy_per_coeff)), energy_per_coeff, color="steelblue")
plt.axvspan(-0.5, 12.5, color="green", alpha=0.15, label="保留 k=0..12（n_mfcc=13，含 k=0）")
plt.xlabel("DCT 系数索引 k")
plt.ylabel("平均能量（均方）")
plt.title("各 DCT 系数的平均能量——能量集中在低阶")
plt.legend()
plt.tight_layout()
plt.show()

## 3. 完整流程图

```
原始音频 x
  │
  ▼ STFT（汉宁窗，win_len=1024，hop=256）
功率谱 |X[k]|²   shape: (n_frames, n_fft//2+1)
  │
  ▼ mel_filterbank @   [mel矩阵，shape: (n_mels, n_fft//2+1)]
mel 能量           shape: (n_frames, n_mels)
  │
  ▼ log（power_to_db，top_db=None）
log-mel 谱         shape: (n_frames, n_mels)
  │
  ▼ DCT-II（ortho归一化）
DCT 系数           shape: (n_frames, n_mels)
  │
  ▼ 取 [:, :n_mfcc]    k=0..n_mfcc-1
MFCC               shape: (n_frames, n_mfcc)
```

每一步都是线性变换或逐元素操作，没有可学习参数。整条流水线的超参数只有 `n_fft`、`hop`、`n_mels`、`n_mfcc` 四个。

In [ ]:
# 演示：逐步走一遍流水线，打印每步形状
from aurora.audio.stft import power_spectrogram
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import dct_ii

sr = 16000
n_fft, hop, n_mels, n_mfcc = 1024, 256, 40, 13
x = sine(440, duration=1.0, sample_rate=sr)
print(f"x.shape           = {x.shape}")

power = power_spectrogram(x, n_fft=n_fft, hop_length=hop)
print(f"power.shape       = {power.shape}  ← (n_frames, n_fft//2+1)")

fb = mel_filterbank(n_mels, n_fft, sr)
mel = power @ fb.T
print(f"mel.shape         = {mel.shape}  ← (n_frames, n_mels)")

log_mel = power_to_db(mel, top_db=None)
print(f"log_mel.shape     = {log_mel.shape}")

coeffs = dct_ii(log_mel)
print(f"dct_coeffs.shape  = {coeffs.shape}")

mfcc_out = coeffs[:, :n_mfcc]
print(f"mfcc_out.shape    = {mfcc_out.shape}  ← (n_frames, n_mfcc) 最终输出")

## 4. ✏️ 实现 `my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)`

**四步串联**：

| 步骤 | 代码 | 输出 shape |
|---|---|---|
| 1 | `S = mel_spectrogram(x, sr, n_fft=win_len, ...)` | `(n_frames, n_mels)` — 功率 Mel 谱 |
| 2 | `logS = power_to_db(S, top_db=None)` | `(n_frames, n_mels)` — 10·log10，内置 1e-10 下限保护 |
| 3 | `coeffs = dct_ii(logS, norm="ortho")` | `(n_frames, n_mels)` — 沿最后一维逐帧变换 |
| 4 | `return coeffs[:, :n_mfcc]` | `(n_frames, n_mfcc)` |

**验收标准**：
- `np.allclose(my_mfcc(x, sr), aurora_mfcc(x, sr), atol=1e-5)` 通过
- 输出 shape `(n_frames, n_mfcc)` = `(n_frames, 13)`
- 无 nan/inf（`power_to_db` 的 `amin=1e-10` 下限保护有效）

> 注意：对数一步必须用 `power_to_db`（10·log10），而不是 `np.log`——两者差一个 10/ln10 ≈ 4.34 的因子，用错会导致与 `aurora_mfcc` 数值对不上。
> 可用导入：`from aurora.audio.mel import mel_spectrogram, power_to_db`，`from aurora.audio.mfcc import dct_ii`

## 4a. 进入 L50 前的 20 秒自检

下面这组不是新知识，只是把前面几课的依赖显式列出来。
先把每项写成 `True/False`，不确定就先回到对应课节。

In [ ]:
pipeline_checklist = {
    "stft_shape_ok": None,  # 卡住回 L44：先确认 STFT 输出 (T, n_bins)
    "mel_matmul_ok": None,  # 卡住回 L47：先确认 power @ fb.T 的方向
    "log_floor_ok": None,   # 卡住回 L47：先想清为什么 log(max(power, eps)) 需要 eps
    "dct_ortho_ok": None,   # 卡住回 L49：回看 DCT-II 的正交归一化
}

pending = [name for name, value in pipeline_checklist.items() if value is None]
print("把 None 改成 True/False；如果不确定，先回看对应课节。")
print("待填写：", pending)

In [ ]:
def my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256):
    """MFCC 提取：x -> (n_frames, n_mfcc)"""
    from aurora.audio.mel import mel_spectrogram, power_to_db
    from aurora.audio.mfcc import dct_ii

    # ✏️ TODO 1: 计算 mel 功率谱，shape (n_frames, n_mels)
    raise NotImplementedError("TODO: mel_spectrogram(x, sr, n_mels=n_mels, n_fft=win_len, hop_length=hop)")

    # ✏️ TODO 2: 取对数（power_to_db，top_db=None），shape 不变
    raise NotImplementedError("TODO: power_to_db(mel, top_db=None)")

    # ✏️ TODO 3: 对每帧做 DCT-II（norm="ortho"），shape (n_frames, n_mels)
    raise NotImplementedError("TODO: dct_ii 对每帧 log_mel[i] 做 DCT-II")

    # ✏️ TODO 4: 取前 n_mfcc 个系数：[:, :n_mfcc]，shape (n_frames, n_mfcc)
    raise NotImplementedError("TODO: 完整 MFCC 流水线：STFT→Mel→log→DCT")

In [ ]:
from aurora.audio.mfcc import mfcc as aurora_mfcc

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)

try:
    out = my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)
except (NotImplementedError, TypeError):
    out = None

if out is None:
    print('⬜ my_mfcc 未实现，请完成四个 TODO 再运行')
else:
    ref = aurora_mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=40)
    print(f'my_mfcc shape : {out.shape}')
    print(f'aurora  shape : {ref.shape}')
    assert out.shape == ref.shape, f'形状不匹配: {out.shape} vs {ref.shape}'
    assert np.allclose(out, ref, atol=1e-5), '数值不一致，请检查 TODO 步骤'
    print('✅ 形状和数值均与 aurora.audio.mfcc.mfcc() 一致')


## 5. 参数实验：MFCC 热图

**实验目标**：可视化 13 个系数随时间的变化。

- **横轴**：帧编号（时间）
- **纵轴**：系数编号 1–13（越低 → 越粗粒度的谱形状）
- **预期现象**：
  - 系数 1（MFCC-1）变化相对平滑，代表整体音色包络
  - 系数 12–13 变化最快，携带细粒度谱纹理信息
  - 对纯正弦波，高阶系数接近零（频谱很纯净）；尝试换成 `chirp` 信号观察差异

修改 `n_mfcc=20` 后，图的纵轴延伸，可以看到 20 阶之后能量快速衰减。

In [ ]:
from aurora.audio.io import chirp

sr = 16000
x_chirp = chirp(f0=200, f1=4000, duration=2.0, sample_rate=sr)

try:
    mfcc_mat = my_mfcc(x_chirp, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)
except (NotImplementedError, TypeError):
    mfcc_mat = None

if mfcc_mat is None:
    print('⬜ 请先完成上方的 my_mfcc TODO，再运行本格')
else:
    import matplotlib.pyplot as plt, numpy as np
    plt.figure(figsize=(10, 4))
    plt.imshow(
        mfcc_mat.T,
        aspect='auto', origin='lower', cmap='magma',
    )
    plt.colorbar(label='MFCC 值')
    plt.xlabel('帧编号'); plt.ylabel('系数编号 k')
    plt.title('Chirp 信号的 MFCC 热图（频率从 200→4000 Hz）')
    plt.tight_layout(); plt.show()
    print(f'✅ mfcc_mat.shape = {mfcc_mat.shape}')


## 🎯 未来的回报 (Future Payoff)
今天你把 STFT → Mel → log → DCT 收束成一个 `my_mfcc()`，这 13 维特征会在 **L51（真实 WAV 实战）/ L62（关键词识别数据集）/ L70（ASR 声学特征）** 再次出现——那时它不再是练习，而是喂给神经网络的**真实输入**。你现在亲手调通的每一步 shape，就是后面模型能跑起来的前提。

## 本课收束

`my_mfcc()` 串联了 `mel_spectrogram → power_to_db → dct_ii → [:n_mfcc]` 四步，输出形状 `(n_frames, n_mfcc)` 的 float64 数组，数值与 `aurora.audio.mfcc.mfcc()` 完全对齐（atol=1e-5）。该函数是 `aurora.audio.mfcc` 模块的核心，Month 2 关键词识别分类器将直接以 `(n_frames, 13)` 的 MFCC 矩阵作为输入特征。下一课（L51）将把这条流水线搬到真实（或合成）WAV 音频上：读文件、四层可视化（波形 / 频谱图 / Mel / MFCC），并可选与 librosa 对答案。

## ✏️ 闭卷推导检查格 — MFCC 流水线维度追踪

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：给定参数：
- 信号长度 `L = 16000`（1 秒，16 kHz）
- `n_fft = 1024`，`hop = 256`，`n_mels = 40`，`n_mfcc = 13`
- aurora 的 `stft` 默认 `center=True`（两端各反射填充 `n_fft//2`），因此 `n_frames = 1 + L // hop`

逐步写出每一步输出的形状：

| 步骤 | 操作 | 输出形状 |
|------|------|---------|
| 1 | 信号 | `(L,)` = (___,) |
| 2 | STFT → magnitude | (___,  ___) |
| 3 | Mel 滤波 | (___, ___) |
| 4 | log（dB） | (___, ___) |
| 5 | DCT-II → MFCC | (___, ___) |

（在此处填表...）

In [ ]:
# 验证：逐步输出形状与推导一致
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.stft import stft
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import mfcc, dct_ii

SR, N_FFT, HOP, N_MELS, N_MFCC = 16000, 1024, 256, 40, 13
signal = np.sin(2*np.pi*440*np.arange(SR)/SR).astype(np.float32)

spec   = stft(signal, n_fft=N_FFT, hop_length=HOP)      # (n_frames, n_bins)
n_frames, n_bins = spec.shape
fb     = mel_filterbank(n_mels=N_MELS, n_fft=N_FFT, sample_rate=SR)
mel_e  = (np.abs(spec)**2) @ fb.T                        # (n_frames, N_MELS)
log_m  = power_to_db(mel_e)                              # (n_frames, N_MELS)
coeffs = mfcc(signal, sample_rate=SR, n_mfcc=N_MFCC,
              n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS) # (n_frames, N_MFCC)

shapes = [spec.shape, mel_e.shape, log_m.shape, coeffs.shape]
labels = ['STFT', 'Mel', 'log-Mel', 'MFCC']
for l, s in zip(labels, shapes): print(f'  {l}: {s}')
assert coeffs.shape == (n_frames, N_MFCC), f'最终形状 {coeffs.shape} 不符预期'
print('✅ MFCC 流水线维度全部验证通过')

In [ ]:
# ✏️ 本课自评
l50_review = {
    "pipeline_5steps":         None,  # 记住 STFT→Mel→log→DCT→截断 五步？True/False
    "my_mfcc_implemented":     None,  # my_mfcc 实现并通过 atol=1e-5 对比？True/False
    "shape_tracking":          None,  # 能手算每步输出 shape？True/False
    "n_mfcc_13_reason":        None,  # 理解前 13 维=声道形状（元音），高维=噪声？True/False
    "whiteboard_passed":       None,  # 白板挑战维度推导闭卷通过？True/False
}

unfilled = [k for k, v in l50_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l50_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L50 全部通关！进入 L51：MFCC 工程实战')

---

→ **下一课**　[L51 · MFCC 工程实战](L51_real_audio.ipynb)

> 下节课将学习 **MFCC 工程实战**：在真实 WAV 音频上提取特征，librosa 对答案。